# Présidentielles françaises 2002-2022 : analyse et tendance 2027

Notebook principal du projet. Il suit le processus KDD :
1. Sélection des données
2. Prétraitement
3. Transformation
4. Fouille (analyse, tendance 2027)
5. Interprétation

Le code est organisé en fonctions, appelées depuis `main()` à la fin du notebook.
Les données brutes doivent être placées dans `data/raw/` (voir le README).

In [ ]:
from pathlib import Path
import json
import unicodedata

import pandas as pd
import plotly.express as px
import pyarrow.parquet as pq

DOSSIER_PROJET = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DOSSIER_RAW = DOSSIER_PROJET / "data" / "raw"
DOSSIER_REFERENCE = DOSSIER_PROJET / "data" / "reference"
DOSSIER_PROCESSED = DOSSIER_PROJET / "data" / "processed"

ANNEES = [2002, 2007, 2012, 2017, 2022]
ELECTIONS = [f"{annee}_pres_t{tour}" for annee in ANNEES for tour in (1, 2)]

## 1. Sélection des données

In [ ]:
def charger_resultats_generaux(chemin=DOSSIER_RAW / "general_results.parquet", elections=ELECTIONS):
    """Charge les résultats généraux (inscrits, votants, abstention...) des présidentielles, par bureau de vote."""
    table = pq.read_table(chemin, filters=[("id_election", "in", elections)])
    return table.to_pandas()


def charger_resultats_candidats(chemin=DOSSIER_RAW / "candidats_results.parquet", elections=ELECTIONS):
    """Charge les voix par candidat et par bureau de vote pour les présidentielles."""
    colonnes = ["id_election", "code_departement", "code_commune", "code_bv",
                "nom", "prenom", "voix", "ratio_voix_inscrits", "ratio_voix_exprimes"]
    table = pq.read_table(chemin, columns=colonnes, filters=[("id_election", "in", elections)])
    return table.to_pandas()


def charger_familles(chemin=DOSSIER_REFERENCE / "candidats_familles.csv"):
    """Charge la table qui associe chaque candidat à une famille politique."""
    return pd.read_csv(chemin)

## 2. Prétraitement

In [ ]:
def normaliser_nom(nom):
    """Met un nom en majuscules sans accents, pour comparer les années entre elles."""
    sans_accents = unicodedata.normalize("NFKD", str(nom)).encode("ascii", "ignore").decode()
    return sans_accents.upper().strip()


def ajouter_annee_tour(df):
    """Ajoute les colonnes annee et tour à partir de id_election (ex. 2022_pres_t1)."""
    df = df.copy()
    df["annee"] = df["id_election"].str[:4].astype(int)
    df["tour"] = df["id_election"].str[-1].astype(int)
    return df


def ajouter_familles(candidats, familles):
    """Associe chaque ligne de candidat à sa famille politique (jointure sur l'année et le nom)."""
    familles = familles.assign(cle_nom=familles["nom"].map(normaliser_nom))[["annee", "cle_nom", "famille"]]
    candidats = candidats.assign(cle_nom=candidats["nom"].map(normaliser_nom))
    resultat = candidats.merge(familles, on=["annee", "cle_nom"], how="left").drop(columns="cle_nom")
    manquants = resultat.loc[resultat["famille"].isna(), "nom"].unique()
    if len(manquants) > 0:
        print("Candidats sans famille politique :", list(manquants))
    return resultat

## 3. Transformation

In [ ]:
def agreger_par_departement(generaux):
    """Somme les résultats généraux par élection et par département, puis recalcule les taux."""
    colonnes = ["inscrits", "abstentions", "votants", "blancs", "nuls", "exprimes"]
    dep = (generaux.groupby(["id_election", "annee", "tour", "code_departement", "libelle_departement"])[colonnes]
           .sum().reset_index())
    dep["taux_abstention"] = dep["abstentions"] / dep["inscrits"]
    dep["taux_blancs_nuls"] = (dep["blancs"] + dep["nuls"]) / dep["votants"]
    return dep


def voix_par_famille_departement(candidats, generaux_dep):
    """Calcule la part des exprimés obtenue par chaque famille politique, par élection et par département."""
    voix = (candidats.groupby(["id_election", "annee", "tour", "code_departement", "famille"])["voix"]
            .sum().reset_index())
    voix = voix.merge(generaux_dep[["id_election", "code_departement", "exprimes"]],
                      on=["id_election", "code_departement"], how="left")
    voix["part_exprimes"] = voix["voix"] / voix["exprimes"]
    return voix

## Cartes

Les contours des départements de métropole viennent de [france-geojson](https://github.com/gregoiredavid/france-geojson). L'outre-mer et les Français de l'étranger ne sont pas sur la carte.

In [ ]:
def charger_contours(chemin=DOSSIER_REFERENCE / "departements.geojson"):
    """Charge les contours des départements de métropole (GeoJSON, propriété code = code du département)."""
    with open(chemin, encoding="utf-8") as f:
        return json.load(f)


def carte_departements(df, colonne, titre, contours=None):
    """Dessine une carte choroplèthe des départements pour une colonne de df (indexée par code_departement)."""
    contours = contours or charger_contours()
    codes_metropole = {f["properties"]["code"] for f in contours["features"]}
    df = df[df["code_departement"].isin(codes_metropole)]
    fig = px.choropleth(df, geojson=contours, locations="code_departement",
                        featureidkey="properties.code", color=colonne,
                        hover_name="code_departement", color_continuous_scale="Viridis",
                        title=titre)
    fig.update_geos(fitbounds="locations", visible=False)
    fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
    return fig

## 4. Contrôles

In [ ]:
def resume_national(candidats):
    """Affiche les quatre premiers candidats au niveau national pour chaque élection (contrôle de cohérence)."""
    totaux = candidats.groupby(["id_election", "nom"])["voix"].sum().reset_index()
    return (totaux.sort_values(["id_election", "voix"], ascending=[True, False])
            .groupby("id_election").head(4))

## Main

In [ ]:
def main():
    generaux = ajouter_annee_tour(charger_resultats_generaux())
    candidats = ajouter_annee_tour(charger_resultats_candidats())
    candidats = ajouter_familles(candidats, charger_familles())

    print(f"Résultats généraux : {len(generaux):,} lignes")
    print(f"Résultats candidats : {len(candidats):,} lignes")

    generaux_dep = agreger_par_departement(generaux)
    familles_dep = voix_par_famille_departement(candidats, generaux_dep)

    DOSSIER_PROCESSED.mkdir(parents=True, exist_ok=True)
    generaux_dep.to_parquet(DOSSIER_PROCESSED / "participation_departements.parquet", index=False)
    familles_dep.to_parquet(DOSSIER_PROCESSED / "familles_departements.parquet", index=False)

    display(resume_national(candidats))

    exemple = familles_dep[(familles_dep["id_election"] == "2022_pres_t1")
                           & (familles_dep["famille"] == "extreme_droite")]
    carte_departements(exemple, "part_exprimes",
                       "Extrême droite au 1er tour 2022 (part des exprimés)").show()
    return generaux_dep, familles_dep


if __name__ == "__main__":
    generaux_dep, familles_dep = main()